# BMI Hand Controller
Ground-up rebuild. Each cell has one job. No silent re-definitions.

**Pipeline stages:**
1. Decode weights → verify power frequency
2. Open-loop validation — confirm muscle head works before any control
3. System identification on the latent wrapper
4. Closed-loop MPC tracking
5. Diagnostics

---
## Stage 0 — Imports and config
All tunable constants live here. Nothing is redefined later.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
%matplotlib inline
from IPython.display import clear_output

from BMI_and_Hand import BMI_and_Hand
from GG4 import Brain
from Controllers import AugmentedKalmanFilter, ConstrainedMPC
from system_id import extract_full_orthogonal_model

# ── Reproducibility ───────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Trajectory config ─────────────────────────────────────────────────
SHAPE        = "circle"
SIZE         = 60.0
SPEED        = 0.2
SIM_STEPS    = 4000
WARMUP_STEPS = 75

# x2 for OPEN-LOOP plan (fed directly to muscle head) — needs amplitude ~8
X2_AMPLITUDE = 8.0

# x2 for CLOSED-LOOP plan (must be achievable by the brain via u ∈ [0,1]).
# At u=[0.5,0.5] the brain produces x2 ≈ 3.14 (DC). Max achievable swing ≈ ±2.
X2_DC_CL    = 3.14
X2_AMP_CL   = 2.0

FEEDFORWARD_GAIN = 1.7
MAX_AMP          = 5.0

# ── System ID config ──────────────────────────────────────────────────
N_DOMINANT          = 2
N_HIDDEN            = 1
N_SAMPLES_PER_TRIAL = 400
NUM_TRIALS          = 5
VAF_THRESHOLD       = 0.70

# ── MPC / LQR config ─────────────────────────────────────────────────
MPC_HORIZON  = 20
MPC_Q        = 1000.0
MPC_R        = 10.0
KF_Q_DIST   = 0.05

print("Config loaded.")
print(f"  Trajectory : {SHAPE}, radius={SIZE}, speed={SPEED} rad/step")
print(f"  Sim steps  : {WARMUP_STEPS} warmup + {SIM_STEPS} drawing = {WARMUP_STEPS + SIM_STEPS} total")
print(f"  x2 OL plan : amplitude={X2_AMPLITUDE} (direct muscle head)")
print(f"  x2 CL plan : DC={X2_DC_CL}, amp={X2_AMP_CL} (brain-achievable)")

---
## Stage 1 — Decode weights and verify power frequency

The muscle head detects x2 oscillations by looking at the signal
at lags of exactly `power_half_period_steps` and `power_full_period_steps`.
The x2 signal **must** oscillate at this period in step-space — not in Hz.
We read the correct period directly from the weights file.

In [ ]:
nn_data = np.load("neural_activity_to_muscle_weights.npz")

# ── Power rhythm: read from weights, not derived ──────────────────────
POWER_HALF_PERIOD = int(nn_data['power_half_period_steps'])
POWER_FULL_PERIOD = int(nn_data['power_full_period_steps'])
# cycles-per-step (what np.sin expects as its angular frequency argument)
POWER_CYCLES_PER_STEP = 1.0 / POWER_FULL_PERIOD

print(f"Power half period : {POWER_HALF_PERIOD} steps")
print(f"Power full period : {POWER_FULL_PERIOD} steps")
print(f"Power freq (cyc/step): {POWER_CYCLES_PER_STEP:.6f}")

# ── Steering frequencies: extract via FFT of cos filter kernels ───────
fft_results = np.abs(np.fft.rfft(nn_data['muscle_filter_cos'], axis=1))
fft_freqs   = np.fft.rfftfreq(nn_data['muscle_filter_cos'].shape[1], d=1.0)
STEERING_FREQS = [fft_freqs[np.argmax(fft_results[i])] for i in range(4)]
# Label order matches muscle order: [shoulder+, shoulder-, elbow+, elbow-]
MUSCLE_LABELS  = ['shoulder+', 'shoulder-', 'elbow+', 'elbow-']

print("\nSteering frequencies (cycles/step):")
for label, freq in zip(MUSCLE_LABELS, STEERING_FREQS):
    print(f"  {label:12s}: {freq:.6f}")

# ── W_total: combined projection from 16D neural → 2D latent ─────────
W_TOTAL = nn_data['output_weight'] @ nn_data['encoder_weight']   # (2, 16)

print(f"\nW_total shape: {W_TOTAL.shape}  (maps 16D neural → 2D latent)")

---
## Stage 2 — Open-loop validation

Before building any controller, verify that a hand-crafted latent plan
produces sensible muscle activations and arm movement.

**Pass criteria (check all three):**
- Power gate reaches > 0.9 after warmup
- All four muscle channels activate at some point
- Arm traces a recognisable circle

In [ ]:
def make_latent_plan(steering_freqs, power_cycles_per_step,
                     vel_seq: np.ndarray,
                     warmup_steps: int,
                     horizon: int,
                     feedforward_gain: float = 2.0,
                     max_amp: float = 2) -> np.ndarray:
    """
    Converts a velocity sequence to a (T + horizon, 2) latent plan.

    x1: sum of directional sinusoids weighted by velocity components.
        Positive vx → f_R channel, negative vx → f_L channel, etc.
    x2: oscillates at exactly power_cycles_per_step so the muscle
        head's convolutional detector locks on and drives power to 1.

    The first `warmup_steps` entries in vel_seq are expected to be zero
    (the arm stands still while the conv cache fills and power builds).
    """
    f_R, f_L, f_U, f_D = steering_freqs
    num_steps  = len(vel_seq)
    total      = num_steps + horizon
    t          = np.arange(total)
    plan       = np.zeros((total, 2))

    for k in range(total):
        vx = vel_seq[k, 0] if k < num_steps else 0.0
        vy = vel_seq[k, 1] if k < num_steps else 0.0

        amp_R = np.clip( feedforward_gain * vx, 0, max_amp) if vx > 0 else 0.0
        amp_L = np.clip(-feedforward_gain * vx, 0, max_amp) if vx < 0 else 0.0
        amp_U = np.clip( feedforward_gain * vy, 0, max_amp) if vy > 0 else 0.0
        amp_D = np.clip(-feedforward_gain * vy, 0, max_amp) if vy < 0 else 0.0

        plan[k, 0] = (
            amp_R * np.sin(2 * np.pi * f_R * t[k]) +
            amp_L * np.sin(2 * np.pi * f_L * t[k]) +
            amp_U * np.sin(2 * np.pi * f_U * t[k]) +
            amp_D * np.sin(2 * np.pi * f_D * t[k])
        )
        # x2: pure sinusoid at exactly the period the muscle head expects
        #plan[k, 1] = np.sin(2 * np.pi * power_cycles_per_step * t[k])
        plan[k, 1] = X2_AMPLITUDE * np.sin(2 * np.pi * power_cycles_per_step * t[k])

    return plan


def make_circle_velocity(size, speed, num_steps, warmup_steps):
    """
    Returns (pos_seq, vel_seq) for a circle, with a zero-velocity
    warmup prepended. Uses analytical derivatives — no finite differences.
    """
    omega = speed / size   # rad/step
    t     = np.arange(num_steps)

    vel = np.stack([
        -size * omega * np.sin(omega * t),
         size * omega * np.cos(omega * t)
    ], axis=1)
    pos = np.stack([
        size * np.cos(omega * t) - size,   # offset so start is at (0,0)
        size * np.sin(omega * t)
    ], axis=1)

    warmup_pos = np.tile(pos[0], (warmup_steps, 1))
    warmup_vel = np.zeros((warmup_steps, 2))

    return (
        np.vstack([warmup_pos, pos]),
        np.vstack([warmup_vel, vel])
    )


print("Helper functions defined.")

In [ ]:
# ── Build plan ────────────────────────────────────────────────────────
pos_seq, vel_seq = make_circle_velocity(SIZE, SPEED, SIM_STEPS, WARMUP_STEPS)
latent_plan = make_latent_plan(
    STEERING_FREQS, POWER_CYCLES_PER_STEP,
    vel_seq, WARMUP_STEPS, horizon=MPC_HORIZON,
    feedforward_gain=FEEDFORWARD_GAIN,
    max_amp=MAX_AMP
)
# The drawing-only positions (no warmup) — used for error reporting
drawing_pos = pos_seq[WARMUP_STEPS:]

print(f"vel_seq shape     : {vel_seq.shape}")
print(f"latent_plan shape : {latent_plan.shape}")
print(f"drawing_pos shape : {drawing_pos.shape}")
vx_peak = np.abs(vel_seq[:, 0]).max()
vy_peak = np.abs(vel_seq[:, 1]).max()
print(f"Peak vx: {vx_peak:.4f}, peak vy: {vy_peak:.4f}")
print(f"Peak x1 amplitude at gain {FEEDFORWARD_GAIN}: {FEEDFORWARD_GAIN * vx_peak:.4f}  (MAX_AMP={MAX_AMP})")
print(f"Peak x2 amplitude: {X2_AMPLITUDE}")
# ── Quick sanity: x2 amplitude and frequency ──────────────────────────
x2 = latent_plan[:, 1]
print(f"\nx2 amplitude range: [{x2.min():.3f}, {x2.max():.3f}]  (expect [-1, 1])")
peaks = np.where((x2[1:-1] > x2[:-2]) & (x2[1:-1] > x2[2:]))[0] + 1
if len(peaks) > 1:
    measured_period = np.diff(peaks).mean()
    print(f"x2 measured period: {measured_period:.1f} steps  (expect {POWER_FULL_PERIOD} steps)")
    if abs(measured_period - POWER_FULL_PERIOD) > 2:
        print("  ⚠ Period mismatch — power gate may not saturate.")
    else:
        print("  ✓ Period matches muscle head expectation.")

In [ ]:
def validate_latent_plan(ann, arm_ref, latent_plan, drawing_pos, warmup_steps):
    """
    Open-loop rollout through muscle head → arm.
    Does NOT touch the Brain. Resets ann and arm state before running.
    Returns predicted hand positions and debug dict.
    """
    ann.muscle_head.reset_state(batch_size=1)

    # Save and reset arm so rollout starts from neutral
    saved_sh = arm_ref._shoulder_angle
    saved_el = arm_ref._elbow_angle
    arm_ref._shoulder_angle = 0.0
    arm_ref._elbow_angle    = 0.0

    latent_tensor = torch.tensor(latent_plan, dtype=torch.float32)
    with torch.no_grad():
        muscle_acts_t, debug = ann.muscle_head(latent_tensor, return_debug=True)
    muscle_acts = muscle_acts_t.numpy()

    T = len(latent_plan)
    predicted = np.zeros((T, 2))
    for k in range(T):
        arm_ref.move(*muscle_acts[k])
        predicted[k] = arm_ref.hand_pos

    # Restore arm state
    arm_ref._shoulder_angle = saved_sh
    arm_ref._elbow_angle    = saved_el

    # ── Metrics ───────────────────────────────────────────────────────
    drawing_pred  = predicted[warmup_steps:warmup_steps + len(drawing_pos)]
    pos_error     = np.linalg.norm(drawing_pred - drawing_pos, axis=1)
    rmse          = np.sqrt(np.mean(pos_error**2))
    power_np      = debug['power'].numpy()
    max_power     = power_np[warmup_steps:].max()

    print(f"Open-loop RMSE (drawing phase)  : {rmse:.3f} units")
    print(f"Peak power gate (post-warmup)   : {max_power:.3f}  (target > 0.9)")
    print(f"Mean muscle acts [sh+,sh-,el+,el-]: {muscle_acts[warmup_steps:].mean(axis=0).round(3)}")

    if max_power < 0.9:
        print("  ⚠ Power gate did not saturate — check POWER_CYCLES_PER_STEP or increase WARMUP_STEPS")
    else:
        print("  ✓ Power gate saturated.")

    if muscle_acts[warmup_steps:, :2].max() < 0.05:
        print("  ⚠ Shoulder muscles never fired — steering frequencies may be wrong")
    else:
        print("  ✓ Shoulder muscles active.")

    # ── Plots ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes[0]
    ax.plot(drawing_pos[:, 0], drawing_pos[:, 1], 'k--', alpha=0.5, label='Target')
    ax.plot(drawing_pred[:, 0], drawing_pred[:, 1], 'b-', lw=1.5, label='Open-loop')
    ax.plot(drawing_pred[0, 0], drawing_pred[0, 1], 'go', label='Start')
    ax.set_title('Open-loop path vs target')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.axis('equal'); ax.grid(True); ax.legend()

    ax = axes[1]
    for i, lbl in enumerate(MUSCLE_LABELS):
        ax.plot(muscle_acts[:, i], label=lbl, lw=0.9)
    ax.axvline(warmup_steps, color='gray', linestyle=':', label='warmup end')
    ax.set_title('Muscle activations')
    ax.set_xlabel('Step'); ax.set_ylabel('Activation [0,1]')
    ax.legend(); ax.grid(True)

    ax = axes[2]
    ax.plot(power_np, 'k-', lw=2, label='Power gate')
    sel = debug['selector'].numpy()
    for i in range(4):
        ax.plot(sel[:, i], lw=0.8, alpha=0.7, label=f'Selector {i}')
    ax.axvline(warmup_steps, color='gray', linestyle=':', label='warmup end')
    ax.set_title('Muscle head internals')
    ax.set_xlabel('Step'); ax.legend(); ax.grid(True)

    plt.tight_layout()
    plt.show()

    return predicted, muscle_acts, {k: v.numpy() for k, v in debug.items()}


# ── Instantiate system (used only for its ann and arm here) ───────────
brain_ol  = Brain(random_seed=RANDOM_SEED)
system_ol = BMI_and_Hand(brain_ol)

# ── Run open-loop validation — MUST pass before continuing ────────────
ol_pred, ol_muscles, ol_debug = validate_latent_plan(
    system_ol.ann, system_ol._arm,
    latent_plan, drawing_pos, WARMUP_STEPS
)
print("\nIf power gate < 0.9 or shoulder muscles silent, STOP here and fix POWER_CYCLES_PER_STEP.")

---
## Stage 3 — System identification

Only proceed here if Stage 2 passed.

We wrap the system so the controller sees a 2D output (the W_total
projection of the 16D neural measurement), then identify a linear
state-space model of this wrapped system.

In [6]:
from bmi_controller import LatentBrainWrapper

# Fresh system for system ID — separate from the open-loop instance
np.random.seed(RANDOM_SEED)
brain_id  = Brain(random_seed=RANDOM_SEED)
system_id = BMI_and_Hand(brain_id)
wrapper   = LatentBrainWrapper(system_id, W_TOTAL)

print("Running system identification...")
print(f"  n_dominant={N_DOMINANT}, n_hidden={N_HIDDEN}")
print(f"  {NUM_TRIALS} trials × {N_SAMPLES_PER_TRIAL} samples each")

lds, vaf = extract_full_orthogonal_model(
    wrapper,
    n_dominant=N_DOMINANT,
    n_hidden=N_HIDDEN,
    n_samples_per_trial=N_SAMPLES_PER_TRIAL,
    num_trials=NUM_TRIALS,
    verbose=True
)

A, B, C   = lds.A, lds.B, lds.C
Q_sys, R_sys = lds.Q, lds.R
n_states  = A.shape[0]

print(f"\nVAF            : {vaf:.3f}  (threshold: {VAF_THRESHOLD})")
print(f"State dim      : {n_states}")
print(f"A shape        : {A.shape}")
print(f"B shape        : {B.shape}")
print(f"C shape        : {C.shape}")

assert vaf >= VAF_THRESHOLD, (
    f"VAF {vaf:.3f} below threshold {VAF_THRESHOLD}. "
    "Increase NUM_TRIALS or N_SAMPLES_PER_TRIAL before continuing."
)
print("\n✓ VAF acceptable. Proceeding to controller design.")

    iter   7  ll=-89.080  Δll=+9.961e+00
    iter   8  ll=-84.970  Δll=+4.109e+00
    iter   9  ll=-83.244  Δll=+1.727e+00
    iter  10  ll=-82.413  Δll=+8.302e-01
    iter  11  ll=-81.934  Δll=+4.798e-01
    iter  12  ll=-81.612  Δll=+3.220e-01
    iter  13  ll=-81.376  Δll=+2.360e-01
    iter  14  ll=-81.195  Δll=+1.810e-01
    iter  15  ll=-81.052  Δll=+1.422e-01
    iter  16  ll=-80.939  Δll=+1.132e-01
    iter  17  ll=-80.848  Δll=+9.081e-02
    iter  18  ll=-80.775  Δll=+7.311e-02
    iter  19  ll=-80.717  Δll=+5.884e-02
    iter  20  ll=-80.669  Δll=+4.712e-02
    iter  21  ll=-80.632  Δll=+3.732e-02
    iter  22  ll=-80.603  Δll=+2.901e-02
    iter  23  ll=-80.581  Δll=+2.186e-02
    iter  24  ll=-80.566  Δll=+1.562e-02
    iter  25  ll=-80.555  Δll=+1.011e-02
    iter  26  ll=-80.550  Δll=+5.213e-03
    iter  27  ll=-80.549  Δll=+8.133e-04
    iter  28  ll=-80.553  Δll=-3.162e-03
    iter  29  ll=-80.559  Δll=-6.773e-03
    iter  30  ll=-80.569  Δll=-1.007e-02
  Reached max_it

---
## Stage 4 — Controller design

MPC cost matrices are sized to match the identified state dimension.
No hardcoded eye(2) — always eye(n_states).

In [7]:
# Derive dimensions from identified matrices — never hardcode
n_outputs = C.shape[0]   # dimension of y = C @ x  (= 2, the latent output)
n_inputs  = B.shape[1]   # dimension of u
from Controllers import LQR
estimator = AugmentedKalmanFilter(
    A, B, C, Q_sys, R_sys,
    Q_disturbance=KF_Q_DIST
)

# Q_cost must be (n_outputs, n_outputs) because the MPC cost is computed
# on the OUTPUT error  e = C @ x_pred - y_ref,  not on the state directly.
# R_cost must be (n_inputs, n_inputs).
mpc = ConstrainedMPC(
    A, B, C,
    horizon=MPC_HORIZON,
    Q_cost=np.eye(n_outputs) * MPC_Q,
    R_cost=np.eye(n_inputs)  * MPC_R
)
# Make sure your LQR class definition is pasted in a cell above this!

lqr = LQR(
    A=A, 
    B=B, 
    C=C, 
    Q_cost_y=np.eye(n_outputs) * MPC_Q, 
    R_cost=np.eye(n_inputs) * MPC_R
)

print("LQR Controller initialized successfully.")
print("Controller design:")
print(f"  n_states  : {n_states}")
print(f"  n_outputs : {n_outputs}  ← Q_cost is ({n_outputs}×{n_outputs})")
print(f"  n_inputs  : {n_inputs}   ← R_cost is ({n_inputs}×{n_inputs})")
print(f"  MPC horizon  : {MPC_HORIZON}")
print(f"  Q weight     : {MPC_Q}")
print(f"  R weight     : {MPC_R}")
print(f"  KF Q_dist    : {KF_Q_DIST}")

# Controllability check
from numpy.linalg import matrix_rank
ctrl_mat = np.hstack([np.linalg.matrix_power(A, i) @ B for i in range(n_states)])
rank = matrix_rank(ctrl_mat)
print(f"\nControllability matrix rank: {rank} / {n_states}")
if rank < n_states:
    print("  ⚠ System is not fully controllable.")
else:
    print("  ✓ System is fully controllable.")

LQR Controller initialized successfully.
Controller design:
  n_states  : 3
  n_outputs : 2  ← Q_cost is (2×2)
  n_inputs  : 2   ← R_cost is (2×2)
  MPC horizon  : 20
  Q weight     : 1000.0
  R weight     : 10.0
  KF Q_dist    : 0.05

Controllability matrix rank: 3 / 3
  ✓ System is fully controllable.


---
## Stage 5 — Closed-loop execution

The wrapper, estimator, and MPC all use the same system instance.
The latent_plan index starts at 0 = first warmup step, so the
controller naturally idles during warmup (zero-velocity feedforward)
before the drawing phase begins.

Total steps run = WARMUP_STEPS + SIM_STEPS.

In [8]:
def run_closed_loop(wrapper, estimator, lqr, C, latent_plan, total_steps, warmup_steps):
    """
    Closed-loop LQR tracking using feedforward + feedback.

    x1 target: taken from latent_plan (steering sinusoids at physical velocity).
    x2 target: REPLACED with brain-achievable oscillation (X2_DC_CL ± X2_AMP_CL)
                because the open-loop plan uses x2 ∈ [-8,8] which the brain
                cannot produce — its x2 output is bounded to ≈ [-1.2, 7.5].

    At each step k the controller sees a 2-step window [y_ref[k], y_ref[k+1]]:
      - y_ref[k]   (disturbance-corrected) drives feedback
      - y_ref[k+1] (nominal)               drives feedforward anticipation
    """
    u_prev = np.zeros(lqr.n_inputs)
    history_hand   = np.zeros((total_steps, 2))
    history_latent = np.zeros((total_steps, 2))

    for k in range(total_steps):
        y_k = wrapper.measure()
        history_hand[k]   = wrapper.system.hand_pos
        history_latent[k] = y_k

        x_hat, d_hat = estimator.update(y_k, u_prev)

        # Build 2-step window: x1 from plan, x2 overridden to achievable range
        end    = min(k + 2, len(latent_plan))
        window = latent_plan[k : end].copy()   # (2,2) or (1,2)
        for i in range(len(window)):
            t_k = k + i
            window[i, 1] = (X2_DC_CL
                            + X2_AMP_CL * np.sin(2*np.pi*POWER_CYCLES_PER_STEP*t_k))

        # Disturbance correction on current step only
        window[0] -= d_hat.flatten()

        u_k = lqr.get_input(x_hat.flatten(), window)
        wrapper.next_state(u_k)
        u_prev = u_k

    return history_hand, history_latent

In [9]:
# ── Fresh system for closed-loop ──────────────────────────────────────
np.random.seed(RANDOM_SEED)
brain_cl   = Brain(random_seed=RANDOM_SEED)
system_cl  = BMI_and_Hand(brain_cl)
wrapper_cl = LatentBrainWrapper(system_cl, W_TOTAL)
estimator_cl = AugmentedKalmanFilter(A, B, C, Q_sys, R_sys, Q_disturbance=KF_Q_DIST)

TOTAL_STEPS = WARMUP_STEPS + SIM_STEPS
print(f"Running closed-loop LQR for {TOTAL_STEPS} steps "
      f"({WARMUP_STEPS} warmup + {SIM_STEPS} drawing)...")

history_hand, history_latent = run_closed_loop(
    wrapper_cl, estimator_cl, lqr, C,
    latent_plan, TOTAL_STEPS, WARMUP_STEPS
)
print("Simulation complete.")

Running closed-loop LQR for 4075 steps (75 warmup + 4000 drawing)...
Simulation complete.


---
## Stage 6 — Diagnostics and plots

In [10]:
# =====================================================================
# STAGE 6 — DIAGNOSTICS
# Ground truth = open-loop ANN rollout from Stage 2 (ol_pred).
# This is the physically correct reference: where the arm actually goes
# when the latent plan is fed through the real ANN and arm model with
# no controller. Everything is measured against this.
# =====================================================================

# ── 1. Slice all signals to drawing phase only ────────────────────────
D = slice(WARMUP_STEPS, WARMUP_STEPS + SIM_STEPS)

ann_ref        = ol_pred[D]          # (SIM_STEPS, 2) — ANN ground truth arm path
drawing_hand   = history_hand[D]     # (SIM_STEPS, 2) — closed-loop arm path
drawing_latent = history_latent[D]   # (SIM_STEPS, 2) — closed-loop latent measurement
plan_drawing   = latent_plan[D]      # (SIM_STEPS, 2) — feedforward latent plan
steps          = np.arange(SIM_STEPS)

# ── 2. Metrics ────────────────────────────────────────────────────────
pos_error   = np.linalg.norm(drawing_hand - ann_ref, axis=1)
rmse        = np.sqrt(np.mean(pos_error**2))
ol_range_x  = ann_ref[:, 0].max() - ann_ref[:, 0].min()
ol_range_y  = ann_ref[:, 1].max() - ann_ref[:, 1].min()
rmse_pct    = rmse / np.mean([ol_range_x, ol_range_y]) * 100

x1_err = np.abs(drawing_latent[:, 0] - plan_drawing[:, 0])
x2_err = np.abs(drawing_latent[:, 1] - plan_drawing[:, 1])

print('══ Arm path metrics (vs ANN open-loop ground truth) ══')
print(f'  RMSE                : {rmse:.3f} arm units')
print(f'  Max error           : {pos_error.max():.3f} arm units')
print(f'  Mean error          : {pos_error.mean():.3f} arm units')
print(f'  RMSE as % of range  : {rmse_pct:.1f}%  '
      f'(X span={ol_range_x:.1f}, Y span={ol_range_y:.1f})')
print()
print('══ Latent tracking metrics ══')
print(f'  x1 mean abs error   : {x1_err.mean():.4f}')
print(f'  x2 mean abs error   : {x2_err.mean():.4f}')
print(f'  x2 actual mean      : {drawing_latent[:, 1].mean():.4f}  '
      f'(plan mean: {plan_drawing[:, 1].mean():.4f})')

# ── 3. Figure 1: Arm paths in physical space ──────────────────────────
# The only meaningful comparison: where did the arm actually go (magenta)
# vs where it would have gone with perfect open-loop (black dashed).
# drawing_pos is NOT used here — it lives in a different coordinate frame.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(history_hand[:, 0], history_hand[:, 1],
        color='lightgray', lw=0.8, zorder=1,
        label='Full run incl. warmup')
ax.plot(ann_ref[:, 0], ann_ref[:, 1],
        'k--', lw=2.0, alpha=0.8, zorder=2,
        label='Open-loop ANN (ground truth)')
ax.plot(drawing_hand[:, 0], drawing_hand[:, 1],
        color='mediumorchid', lw=2.0, zorder=3,
        label='Closed-loop actual')
ax.plot(drawing_hand[0, 0],  drawing_hand[0, 1],
        'go', ms=10, zorder=4, label='Start')
ax.plot(drawing_hand[-1, 0], drawing_hand[-1, 1],
        'rs', ms=10, zorder=4, label='End')
ax.set_title(f'Arm path — closed-loop vs ANN ground truth\n'
             f'RMSE = {rmse:.2f} arm units ({rmse_pct:.1f}% of workspace)')
ax.set_xlabel('X (arm units)')
ax.set_ylabel('Y (arm units)')
ax.axis('equal'); ax.grid(True)
ax.legend(fontsize=8, loc='best')

ax = axes[1]
ax.plot(steps, pos_error, color='crimson', lw=1.2, label='Tracking error')
ax.fill_between(steps, pos_error, alpha=0.15, color='crimson')
ax.axhline(rmse, color='k', linestyle='--', lw=1.5,
           label=f'RMSE = {rmse:.2f}')
ax.set_title('Arm tracking error vs ANN ground truth')
ax.set_xlabel('Step (drawing phase only)')
ax.set_ylabel('Euclidean error (arm units)')
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

# ── 4. Figure 2: Latent signals ───────────────────────────────────────
# Shows whether the MPC successfully drove the brain to produce the
# intended x1 (steering) and x2 (power) signals.
# x2 not tracking the plan = power gate being suppressed by the MPC.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(steps, plan_drawing[:, 0],
        'k--', lw=0.8, alpha=0.5, label='Plan x1 (feedforward)')
ax.plot(steps, drawing_latent[:, 0],
        color='steelblue', lw=1.0, label='Actual x1')
ax.set_title(f'Latent x1 — steering signal\n'
             f'mean abs error = {x1_err.mean():.4f}')
ax.set_xlabel('Step (drawing phase only)')
ax.set_ylabel('x1 activation')
ax.legend(); ax.grid(True)

ax = axes[1]
ax.plot(steps, plan_drawing[:, 1],
        'k--', lw=0.8, alpha=0.5, label='Plan x2 (power)')
ax.plot(steps, drawing_latent[:, 1],
        color='tomato', lw=1.0, label='Actual x2')
ax.set_title(f'Latent x2 — power signal\n'
             f'mean abs error = {x2_err.mean():.4f}')
ax.set_xlabel('Step (drawing phase only)')
ax.set_ylabel('x2 activation')
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

══ Arm path metrics (vs ANN open-loop ground truth) ══
  RMSE                : 81.749 arm units
  Max error           : 119.994 arm units
  Mean error          : 75.055 arm units
  RMSE as % of range  : 68.1%  (X span=120.0, Y span=120.0)

══ Latent tracking metrics ══
  x1 mean abs error   : 0.7475
  x2 mean abs error   : 5.1663
  x2 actual mean      : 0.0140  (plan mean: 0.0032)


NExt step:

trajectory tracking

In [11]:
def get_velocity_command(current_pos, target_pos):
    """Proportional nav toward target, clamped to [-2, 2]."""
    vec  = np.array(target_pos) - np.array(current_pos)
    dist = np.linalg.norm(vec)
    return np.clip(0.5 * vec, -2.0, 2.0) if dist > 1.0 else np.zeros(2)


def map_velocity_to_latent(velocity, t, steering_freqs, f_power):
    """
    Convert velocity command at global time t to (x1, x2).

    x1: frequency-multiplexed directional sinusoids.
    x2: brain-achievable power oscillation centred at X2_DC_CL.
        Uses X2_DC_CL / X2_AMP_CL (not the open-loop X2_AMPLITUDE=8)
        because the brain's x2 output is bounded to ≈ [-1.2, 7.5].

    t must never reset between targets so the power rhythm stays phase-coherent.
    """
    vx, vy  = velocity
    f_R, f_L, f_U, f_D = steering_freqs
    moving  = np.linalg.norm(velocity) > 0.1

    amp_R = np.clip( FEEDFORWARD_GAIN * vx, 0, MAX_AMP) if vx > 0 else 0.0
    amp_L = np.clip(-FEEDFORWARD_GAIN * vx, 0, MAX_AMP) if vx < 0 else 0.0
    amp_U = np.clip( FEEDFORWARD_GAIN * vy, 0, MAX_AMP) if vy > 0 else 0.0
    amp_D = np.clip(-FEEDFORWARD_GAIN * vy, 0, MAX_AMP) if vy < 0 else 0.0

    x1 = (amp_R * np.sin(2*np.pi*f_R*t) + amp_L * np.sin(2*np.pi*f_L*t) +
          amp_U * np.sin(2*np.pi*f_U*t) + amp_D * np.sin(2*np.pi*f_D*t))

    # Power oscillation in achievable range; stays at DC when arm is stationary
    x2 = (X2_DC_CL + X2_AMP_CL * np.sin(2*np.pi*f_power*t)) if moving else X2_DC_CL
    return np.array([x1, x2])

In [ ]:
def run_real_time_reaching(wrapper, lqr, estimator, steering_freqs, power_freq, targets):
    """
    Real-time reaching displayed inline via clear_output (no GUI window).
    Redraws every 10 steps by replacing the cell output in place.
    """
    history_hand = []
    u_prev   = np.zeros(2)
    t_global = 0

    for target in targets:
        print(f"Reaching for target: {target}")
        for _ in range(500):
            y_k      = wrapper.measure()
            curr_pos = wrapper.system.hand_pos
            history_hand.append(curr_pos.copy())

            x_hat, d_hat = estimator.update(y_k, u_prev)

            vel = get_velocity_command(curr_pos, target)
            y_ref_k  = map_velocity_to_latent(vel, t_global,     steering_freqs, power_freq)
            y_ref_k1 = map_velocity_to_latent(vel, t_global + 1, steering_freqs, power_freq)

            window = np.vstack([
                y_ref_k  - d_hat.flatten(),
                y_ref_k1,
            ])

            u_k = lqr.get_input(x_hat.flatten(), window)
            wrapper.next_state(u_k)
            u_prev    = u_k
            t_global += 1

            if t_global % 10 == 0:
                clear_output(wait=True)
                fig, ax = plt.subplots(figsize=(6, 6))
                ax.set_xlim(-150, 150); ax.set_ylim(-150, 150)

                trace = np.array(history_hand)
                ax.plot(trace[:, 0], trace[:, 1], 'm-', alpha=0.4, lw=1, label='Path')

                sh = float(wrapper.system._arm._shoulder_angle)
                el = float(wrapper.system._arm._elbow_angle)
                ex, ey = 60*np.cos(sh), 60*np.sin(sh)
                hx, hy = ex + 60*np.cos(sh+el), ey + 60*np.sin(sh+el)
                ax.plot([0, ex, hx], [0, ey, hy], 'o-', lw=4, color='navy', label='Arm')

                ax.plot(*target, 'r*', ms=15, label='Target')
                ax.set_title(f'Step {t_global}  |  target {target}')
                ax.legend(loc='upper right', fontsize=8)
                ax.set_aspect('equal')
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
                plt.close(fig)

    return np.array(history_hand)


# ── Fresh system for reaching ─────────────────────────────────────────
np.random.seed(RANDOM_SEED)
brain_rt   = Brain(random_seed=RANDOM_SEED)
system_rt  = BMI_and_Hand(brain_rt)
wrapper_rt = LatentBrainWrapper(system_rt, W_TOTAL)
estimator_rt = AugmentedKalmanFilter(A, B, C, Q_sys, R_sys, Q_disturbance=KF_Q_DIST)

my_targets = [(60.0, 60.0), (-40.0, 30.0), (20.0, -50.0)]
reaching_history = run_real_time_reaching(
    wrapper_rt, lqr, estimator_rt,
    STEERING_FREQS, POWER_CYCLES_PER_STEP,
    my_targets
)